# 第6章 用 rocprof 找到慢在哪里

**操作手册** | 对照两个 vector add，只看 kernel 时间、工作划分和 stride 趋势

本手册对应文档：`docs/part1-profiling/chapter6/index.md`  
本手册对应代码：`code/part1-profiling/chapter6/`

---

### 本章导读

> 本章只解决一个问题：**两个 vector add 实现速度差很多时，怎么先找到慢在哪个 kernel？**
>
> 我们会先用 benchmark 看差距，再用 `rocprofv3` 查看每次 kernel dispatch，最后扫描 `stride` 观察变化趋势。这个案例还会提醒你：命令行里只改一个参数，不代表 GPU 内部只改了一件事。读完后，你应该会定位慢点，也知道什么时候还不能急着下结论。

本章代码在 `code/part1-profiling/chapter6/vector_add.hip`。下面的性能数据来自 **Radeon RX 9070 XT（gfx1201）+ ROCm 7.13 + 原生 Ubuntu 24.04**；换一张卡，数字会变，但操作顺序不变。

## Goal

学会用 `rocprofv3` 定位慢 kernel，理解工作划分对性能的影响。具体目标：

1. 用 benchmark 确认两个实现的速度差距
2. 用 `rocprofv3 --kernel-trace` 找到慢在哪个 dispatch
3. 对比 coalesced 和 linecross 的 Grid Size、VGPR、SGPR
4. 扫描 stride 参数，观察性能趋势
5. 识别实验同时改变了哪些变量（地址排布、循环次数、Grid Size）

## Prerequisite

- 已完成第5章 benchmark 与可信计时
- ROCm 环境已激活（`source code/part1-profiling/activate-rocm.sh`）
- `rocprofv3` 可用（ROCm 7.13+）
- 理解 warmup、repeat、GPU event 计时

## Platform

本手册基于以下环境验证：

- **GPU**: AMD Radeon RX 9070 XT (gfx1201)
- **ROCm**: 7.13
- **OS**: Ubuntu 24.04 (native, kernel 6.17.0-35-generic)
- **hipcc**: 7.13.99004 / arch gfx1201

其他 RDNA3/RDNA4 架构（gfx1100, gfx1151, gfx1201）均可运行，数字会有差异。

## 6.1 先看懂两个实现

这一节先看两个 kernel 分别怎样把 `n` 个元素分给线程。它们的输出相同，但线程的工作划分并不相同。

### 6.1.1 连续访存版

普通 vector add 让线程 `i` 处理元素 `i`：

```cpp
__global__ void kernel_coalesced(const float* a, const float* b,
                                 float* c, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) {
        c[i] = a[i] + b[i];
    }
}
```

一个 wavefront 里的 lane 0、lane 1、lane 2 会依次访问 `a[0]`、`a[1]`、`a[2]`。这些地址连在一起，GPU 可以把多条线程请求合并成较少的内存事务。这就是**合并访存（Memory Coalescing）**。

这个版本每个线程只计算一个输出，因此一共启动约 `n` 个线程。

### 6.1.2 linecross 版

`linecross` 是本章给对照实现起的名字，不是 ROCm 的标准术语。它让每个 lane 处理一小段连续元素：

```cpp
int tile_base = wave_id * 32 * stride;
int my_start = tile_base + lane * stride;
for (int j = 0; j < stride; ++j) {
    int i = my_start + j;
    c[i] = a[i] + b[i];
}
```

以 `stride=32` 为例，同一次循环中，各 lane 看到的地址大致是：

```text
lane 0  -> a[0]
lane 1  -> a[32]
lane 2  -> a[64]
lane 3  -> a[96]
...
```

相邻 lane 的起点隔了 32 个 float，也就是 128 字节。地址排布比连续访存版更分散。

一个 wavefront 一共处理 `32 × stride` 个输出，所以 stride 变大时会同时发生两件事：

1. 每个 lane 的循环次数增加；
2. 需要启动的 wavefront 数减少。

*（图示：两个实现同时改变了地址排布和线程工作划分。连续访存版——lane 地址相邻、每个线程 1 个输出、启动约 n 个线程；linecross stride=32——lane 起点相隔 128 字节、每个线程 32 个输出、启动约 n/32 个线程。）*

这不是一个"只改地址排布"的严格对照。它适合练习 benchmark 和 kernel trace，也能展示 stride 增大时的整体趋势；但仅凭这组数据，不能把全部性能差距都归因于访存合并。

## Parameter

### vector_add_bench 参数

| 参数 | 含义 | 示例值 |
|------|------|--------|
| `--kernel` | 选择 coalesced 或 linecross | coalesced |
| `--size` | 处理元素个数 | 16777216 (16M) |
| `--block` | 每个 block 的线程数 | 256 |
| `--stride` | linecross 中每个 lane 负责的连续元素数 | 1, 32 |
| `--warmup` | 热身次数 | 20 |
| `--repeat` | 正式计时次数 | 100 |
| `--output-json` | 保存结果到 JSON | logs/result.json |

### 有效带宽的计算

程序会同时输出延迟和**有效带宽**。按算法口径，vector add 每个元素需要读 `a`、读 `b`、写 `c`，合计 12 B 有效数据，因此：

```text
有效带宽 = 12 × 元素个数 / kernel 时间
```

有效带宽是为了方便比较而换算出的数值，不等于硬件实际发出的 DRAM 事务量。

### rocprofv3 参数

| 参数 | 含义 |
|------|------|
| `--kernel-trace` | 收集 kernel dispatch 跟踪 |
| `-o <file>` | 输出文件路径 |
| `-f csv` | 输出格式为 CSV |

## Execution

### 步骤1：定位仓库根目录并进入工作目录

In [ ]:
import os
import subprocess
from pathlib import Path

# 定位仓库根目录
REPO_ROOT = Path.cwd().resolve().parents[1] if "notebooks" in str(Path.cwd()) else Path.cwd()
WORK_DIR = REPO_ROOT / "code/part1-profiling/chapter6"
os.chdir(WORK_DIR)

print(f"仓库根目录: {REPO_ROOT}")
print(f"工作目录: {WORK_DIR}")
print(f"当前目录: {Path.cwd()}")

### 步骤2：检测 GPU 架构

在编译前自动检测当前 GPU 架构（gfx1100/gfx1151/gfx1201），避免硬编码特定架构。

In [ ]:
# 检测 GPU 架构（gfx1100/gfx1151/gfx1201）
import subprocess

def detect_gpu_arch():
    """检测当前 GPU 架构，返回 gfxXXXX 字符串"""
    try:
        # 方法1: 尝试 rocminfo
        result = subprocess.run(
            ["rocminfo"],
            capture_output=True,
            text=True,
            timeout=10
        )
        if result.returncode == 0:
            for line in result.stdout.split('\n'):
                if 'Name:' in line and 'gfx' in line.lower():
                    # 提取 gfxXXXX
                    parts = line.split()
                    for part in parts:
                        if part.startswith('gfx'):
                            return part
    except (FileNotFoundError, subprocess.TimeoutExpired):
        pass
    
    try:
        # 方法2: 尝试 rocm-smi --showproductname
        result = subprocess.run(
            ["rocm-smi", "--showproductname"],
            capture_output=True,
            text=True,
            timeout=10
        )
        if result.returncode == 0:
            output = result.stdout.lower()
            # 根据产品名推断架构
            if '9070' in output or '9060' in output:
                return 'gfx1201'  # RDNA4
            elif '7900' in output or '7800' in output or '7700' in output:
                return 'gfx1100'  # RDNA3
    except (FileNotFoundError, subprocess.TimeoutExpired):
        pass
    
    # 架构切换说明：自动检测失败时，按目标 GPU 选择回退示例。
# gfx1100（RDNA 3 独立显卡）、gfx1151（Ryzen AI / RDNA 3.5）、gfx1201（RX 9070 XT / RDNA 4）。
# 下面使用项目基线 gfx1201；其他 GPU 请把返回值改成对应 gfx 架构。
    print("⚠ 无法自动检测架构，使用默认值 gfx1201")
    return 'gfx1201'

# 检测并验证架构
GPU_ARCH = detect_gpu_arch()
# 后续 hipcc 通过 --offload-arch={GPU_ARCH} 使用该值，不需要修改编译命令。
print(f"检测到 GPU 架构: {GPU_ARCH}")

# 验证架构是否在支持列表中
SUPPORTED_ARCHS = ['gfx1100', 'gfx1151', 'gfx1201']
if GPU_ARCH not in SUPPORTED_ARCHS:
    print(f"⚠ 警告: {GPU_ARCH} 不在已验证列表 {SUPPORTED_ARCHS} 中")
    print(f"  继续使用 {GPU_ARCH}，但结果可能与文档基线不同")
else:
    print(f"✓ 架构 {GPU_ARCH} 已验证（支持 RDNA3/RDNA4）")


## 6.2 先跑一遍，确认谁更慢

这一节先不打开 profiler，只用第 5 章的计时方法比较三个配置。从仓库根目录进入本篇环境并编译，然后运行连续访存版，再把 `linecross` 的 stride 分别设为 1 和 32。

### 步骤3：编译 vector_add_bench

编译 HIP 程序，生成 benchmark 可执行文件：

In [ ]:
# 创建 logs 目录
logs_dir = WORK_DIR / "logs"
logs_dir.mkdir(exist_ok=True)

# 编译 vector_add.hip（使用检测到的架构）
source_file = WORK_DIR / "vector_add.hip"
output_binary = WORK_DIR / "vector_add_bench"

if source_file.exists():
    print(f"编译: {source_file.name}")
    compile_cmd = [
        "hipcc",
        f"--offload-arch={GPU_ARCH}",  # 使用检测到的架构
        "-O3",
        str(source_file),
        "-o",
        str(output_binary)
    ]
    print(f"编译命令: {' '.join(compile_cmd)}")
    result = subprocess.run(compile_cmd, capture_output=True, text=True)
    if result.returncode == 0:
        print(f"编译成功: {output_binary}")
    else:
        print(f"编译失败: {result.stderr}")
else:
    print(f"未找到源文件: {source_file}")


### 步骤4：运行 benchmark — coalesced 版本

In [ ]:
if output_binary.exists():
    print("运行 coalesced 版本:")
    cmd = [
        str(output_binary),
        "--kernel", "coalesced",
        "--size", "16777216",
        "--block", "256",
        "--warmup", "20",
        "--repeat", "100",
        "--output-json", str(logs_dir / "coalesced_size16777216.json")
    ]
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=60)
    print(result.stdout)
    if result.returncode != 0:
        print(f"错误: {result.stderr}")
else:
    print(f"未找到可执行文件: {output_binary}")

### 步骤5：运行 benchmark — linecross stride=1

`linecross stride=1` 每个线程也只处理一个元素，线程数和连续访存版相同，因此两者时间应该接近。

In [ ]:
if output_binary.exists():
    print("运行 linecross stride=1:")
    cmd = [
        str(output_binary),
        "--kernel", "linecross",
        "--size", "16777216",
        "--block", "256",
        "--stride", "1",
        "--warmup", "20",
        "--repeat", "100",
        "--output-json", str(logs_dir / "linecross_stride1_size16777216.json")
    ]
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=60)
    print(result.stdout)
    if result.returncode != 0:
        print(f"错误: {result.stderr}")

### 步骤6：运行 benchmark — linecross stride=32

到了 `stride=32`，每个线程处理 32 个元素，Grid Size 也会缩小到原来的 1/32。

In [ ]:
if output_binary.exists():
    print("运行 linecross stride=32:")
    cmd = [
        str(output_binary),
        "--kernel", "linecross",
        "--size", "16777216",
        "--block", "256",
        "--stride", "32",
        "--warmup", "20",
        "--repeat", "100",
        "--output-json", str(logs_dir / "linecross_stride32_size16777216.json")
    ]
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=60)
    print(result.stdout)
    if result.returncode != 0:
        print(f"错误: {result.stderr}")

### benchmark 结果解读

在 9070XT 上得到的参考结果如下：

| kernel | stride | 最短时间 | 有效带宽 | 正确性 |
|--------|--------|----------|----------|--------|
| coalesced | - | 0.334 ms | 603 GB/s | OK |
| linecross | 1 | 0.336 ms | 599 GB/s | OK |
| linecross | 32 | 2.25 ms | 89.7 GB/s | OK |

`linecross stride=1` 每个线程也只处理一个元素，线程数和连续访存版相同，因此两者时间接近。

到了 `stride=32`，时间增加到 2.25 ms，约为连续访存版的 **6.7 倍**。现在可以确认这个配置更慢，但还不能确认是地址分散、wavefront 变少，还是两者共同造成。

## 6.3 用 rocprof 看每次 kernel dispatch

这一节只用 `rocprofv3` 的 kernel trace（核函数跟踪），不碰复杂计数器。GPU event 已经给出了计时结果，kernel trace 的新增价值是把每次 dispatch 单独列出来；以后面对包含很多 kernel 的程序，就能用它找到最慢的那一个。

程序一共启动 15 次 kernel：前 5 次是 warmup，后 10 次才是正式结果。第一次打开生成的 CSV，先找下面几组列：

| 列 | 先用它回答什么 |
|------|------|
| `Kernel_Name` | 到底运行了哪个 kernel |
| `Start_Timestamp` / `End_Timestamp` | 单次 kernel 花了多久 |
| `Grid_Size` | 一共启动了多少个 work-item |
| `VGPR_Count` / `SGPR_Count` | kernel 的寄存器分配 |

时间戳单位是纳秒：

```text
kernel 时间（μs）= (End_Timestamp - Start_Timestamp) / 1000
```

### 步骤7：检查 rocprofv3 可用性

在采集 kernel trace 前，先检查 `rocprofv3` 是否可用：

In [ ]:
# 检查 rocprofv3 是否可用
try:
    result = subprocess.run(
        ["rocprofv3", "--version"],
        capture_output=True,
        text=True,
        timeout=5
    )
    if result.returncode == 0:
        print("rocprofv3 可用:")
        print(result.stdout)
        rocprof_available = True
    else:
        print("rocprofv3 不可用")
        rocprof_available = False
except FileNotFoundError:
    print("rocprofv3 未找到，请确认 ROCm 环境已激活")
    rocprof_available = False

### 步骤7（续）：采集 kernel trace（如果 rocprofv3 可用）

使用 `rocprofv3 --kernel-trace` 收集每次 kernel dispatch 的时间戳和配置：

In [ ]:
if rocprof_available and output_binary.exists():
    print("采集 coalesced kernel trace:")
    cmd = [
        "rocprofv3",
        "--kernel-trace",
        "-o", str(logs_dir / "final_kt_coalesced.csv"),
        "-f", "csv",
        "--",
        str(output_binary),
        "--kernel", "coalesced",
        "--size", "16777216",
        "--block", "256",
        "--warmup", "5",
        "--repeat", "10"
    ]
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=60)
    print(result.stdout)
    if result.returncode != 0:
        print(f"错误: {result.stderr}")
    
    print("\n采集 linecross stride=32 kernel trace:")
    cmd = [
        "rocprofv3",
        "--kernel-trace",
        "-o", str(logs_dir / "final_kt_linecross32.csv"),
        "-f", "csv",
        "--",
        str(output_binary),
        "--kernel", "linecross",
        "--size", "16777216",
        "--block", "256",
        "--stride", "32",
        "--warmup", "5",
        "--repeat", "10"
    ]
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=60)
    print(result.stdout)
    if result.returncode != 0:
        print(f"错误: {result.stderr}")
else:
    print("跳过 kernel trace 采集（rocprofv3 不可用或可执行文件不存在）")

### kernel trace 结果解读

跳过最前面的 5 行 warmup，再统计后 10 行。参考运行得到：

| kernel | 单次最短时间 | 中位数 | Grid Size | VGPR | SGPR |
|--------|--------------|--------|-----------|------|------|
| `kernel_coalesced` | 329 μs | 330 μs | 16,777,216 | 8 | 128 |
| `kernel_linecross`（stride=32） | 2202 μs | 2304 μs | 524,288 | 16 | 128 |

这个表先读出三件事：

1. kernel trace 的 329 μs 和 benchmark 的 0.334 ms 基本一致，两种计时方法互相对得上；
2. `kernel_linecross` 是更慢的 dispatch；
3. 它的 Grid Size 只有连续访存版的 1/32，说明线程工作划分确实一起变了。

这就是 kernel trace 的第一价值：**先把“程序慢”缩小成“某个 kernel 慢”，再看这个 kernel 的启动配置。**

## 6.4 先列出一起变化的东西

这一节不增加新工具，只检查实验到底同时改了哪些变量。

从源码和 trace 可以列出：

| 变化 | coalesced | linecross stride=32 |
|------|-----------|---------------------|
| 同时访问的地址 | 相邻 | 更分散 |
| 每个线程处理的输出 | 1 个 | 32 个 |
| Grid Size | 16,777,216 | 524,288 |
| kernel 时间 | 0.334 ms | 2.25 ms |

*（图示：一次改了多件事时，先不要急着把结果归给其中一件。linecross 更慢可能因为地址排布变了、每线程循环次数变了、Grid Size 变了——它们都需要新的公平对照。）*

访存合并是一个合理方向，但并不是当前数据唯一支持的解释。有效带宽从 603 GB/s 降到 89.7 GB/s，也只是同一份时间结果换成了带宽单位，不能算第二份独立测量。

## 6.5 看看静态资源有没有变

这一节检查 VGPR、SGPR 和 LDS。Occupancy（占用率）在这里可以先简单理解成“GPU 能同时保留多少个 wavefront 轮流工作”。

`linecross stride=1` 和 `linecross stride=32` 执行的是同一个编译后的 kernel。stride 是运行时参数，因此两种配置的静态资源分配相同：

| 资源 | linecross stride=1 | linecross stride=32 |
|------|------------------:|-------------------:|
| VGPR | 16 | 16 |
| SGPR | 128 | 128 |
| LDS | 0 | 0 |

这说明寄存器和 LDS 分配不是两个 stride 配置之间的变量。不过，stride 仍然改变了每个线程的循环次数和 Grid Size，所以还不能把剩余差距全部交给访存合并解释。

这一节的结论很窄：**静态资源没变，但工作划分变了。**

### 步骤8：reference fallback — 加载已有证据

当 profiler 不可用时，加载仓库中已有的参考数据：

In [ ]:
import json

# reference fallback: 加载已有的实测数据
reference_data = {
    "platform": "Radeon RX 9070 XT (gfx1201) + ROCm 7.13",
    "measured": {
        "coalesced": {
            "min_time_ms": 0.334,
            "median_time_ms": 0.337,
            "effective_bw_gbps": 603,
            "grid_size": 16777216,
            "vgpr": 8,
            "sgpr": 128
        },
        "linecross_stride1": {
            "min_time_ms": 0.336,
            "median_time_ms": 0.338,
            "effective_bw_gbps": 599,
            "grid_size": 16777216,
            "vgpr": 16,
            "sgpr": 128
        },
        "linecross_stride32": {
            "min_time_ms": 2.25,
            "median_time_ms": 2.30,
            "effective_bw_gbps": 89.7,
            "grid_size": 524288,
            "vgpr": 16,
            "sgpr": 128
        }
    },
    "hypothesis": [
        "linecross stride=32 同时改变了地址排布、每线程循环次数和 Grid Size",
        "Grid Size 从 16M 降到 524K（1/32），说明线程工作划分确实变了",
        "VGPR/SGPR 分配相同（stride 是运行时参数），静态资源不是变量",
        "有效带宽从 603 GB/s 降到 89.7 GB/s（约 6.7 倍差距）"
    ]
}

print("=== Reference Data (measured on 9070XT + ROCm 7.13) ===")
print(json.dumps(reference_data, indent=2, ensure_ascii=False))

## 6.6 用 stride 扫描观察趋势

这一节扫描 `stride`，观察这个 `linecross` 实现的整体性能怎样变化。

### 步骤9：stride 扫描（观察趋势）

扫描多个 stride 值，观察 linecross 实现的整体性能变化：

In [ ]:
if output_binary.exists():
    print("stride 扫描:")
    stride_values = [1, 2, 4, 8, 16, 32, 64, 128, 256]
    
    for s in stride_values:
        cmd = [
            str(output_binary),
            "--kernel", "linecross",
            "--size", "16777216",
            "--block", "256",
            "--stride", str(s),
            "--warmup", "20",
            "--repeat", "50",
            "--output-json", str(logs_dir / f"linecross_s{s}.json")
        ]
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=60)
        # 只打印关键行
        for line in result.stdout.split('\n'):
            if 'stride' in line.lower() or 'time' in line.lower() or 'bandwidth' in line.lower():
                print(line)
else:
    print(f"未找到可执行文件: {output_binary}")
    print("\nreference fallback: stride 扫描参考数据")
    stride_reference = [
        (1, 0.338, 596),
        (8, 0.405, 497),
        (16, 1.65, 122),
        (32, 2.19, 92.1),
        (64, 4.86, 41.4),
        (256, 25.3, 7.95)
    ]
    print(f"{'stride':>6} | {'time_ms':>8} | {'eff_bw_gbps':>12} | {'vs_stride1':>10}")
    print("-" * 45)
    for s, t, bw in stride_reference:
        ratio = t / stride_reference[0][1]
        print(f"{s:>6} | {t:>8.3f} | {bw:>12.2f} | {ratio:>9.2f}x")

### stride 扫描结果解读

参考值摘要：

| stride | 最短时间 | 有效带宽 | 相对 stride=1 耗时 |
|-------:|---------:|---------:|-------------------:|
| 1 | 0.338 ms | 596 GB/s | 1.00× |
| 8 | 0.405 ms | 497 GB/s | 1.20× |
| 16 | 1.65 ms | 122 GB/s | 4.89× |
| 32 | 2.19 ms | 92.1 GB/s | 6.47× |
| 64 | 4.86 ms | 41.4 GB/s | 14.4× |
| 256 | 25.3 ms | 7.95 GB/s | 74.9× |

可以直接观察到：stride 整体越大，这个实现越慢。但一个命令行参数同时改变了地址跨度、每线程循环次数和 Grid Size，所以这条曲线描述的是**组合效果**，不是单独的 cache line 或合并访存曲线。

### 公平对照应该怎么设计

要单独验证访存合并，下一组实验需要固定三件事：

1. 启动相同数量的线程和 wavefront；
2. 每个线程执行相同次数的循环和加法；
3. 只改变循环里的索引公式，让一版地址相邻、另一版地址分散。

例如，两版都让每个 lane 处理 32 个元素，只改变访问顺序：

```text
连续版：i = tile_base + j * 32 + lane
分散版：i = tile_base + lane * 32 + j
```

这才是后续应该补跑的公平对照。在这组新数据产生之前，本章停在“找到慢 kernel，并发现实验同时改变了多个底层变量”这个结论上。

*（图示：本章走完的最小 profiling 路线——benchmark 确认差距 → kernel trace 找到慢 dispatch → 列出所有变化变量 → 设计公平对照 → 再决定优化方向。profiler 不会自动替你证明原因。它先帮你找到慢点；真正解释原因，还需要源码检查和公平对照。）*

## Expected Output / Interpretation

### benchmark 预期输出

在 9070XT (gfx1201) + ROCm 7.13 上，预期看到：

| kernel | stride | 最短时间 | 有效带宽 | 正确性 |
|--------|--------|----------|----------|--------|
| coalesced | - | 0.334 ms | 603 GB/s | OK |
| linecross | 1 | 0.336 ms | 599 GB/s | OK |
| linecross | 32 | 2.25 ms | 89.7 GB/s | OK |

**解读**：
- `linecross stride=1` 与 coalesced 时间接近（每个线程也只处理一个元素）
- `linecross stride=32` 慢约 6.7 倍，但同时改变了地址排布、循环次数和 Grid Size

### kernel trace 预期输出（如果 rocprofv3 可用）

CSV 文件中关键列：

| kernel | 单次最短时间 | 中位数 | Grid Size | VGPR | SGPR |
|--------|--------------|--------|-----------|------|------|
| kernel_coalesced | 329 μs | 330 μs | 16,777,216 | 8 | 128 |
| kernel_linecross (s=32) | 2202 μs | 2304 μs | 524,288 | 16 | 128 |

**解读**：
- kernel trace 的 329 μs 与 benchmark 的 0.334 ms 基本一致
- `kernel_linecross` 的 Grid Size 只有 coalesced 的 1/32
- VGPR/SGPR 分配相同（stride 是运行时参数）

### stride 扫描预期趋势

| stride | 相对 stride=1 耗时 |
|--------|-------------------|
| 1 | 1.00× |
| 8 | 1.20× |
| 16 | 4.89× |
| 32 | 6.47× |
| 64 | 14.4× |
| 256 | 74.9× |

**解读**：
- stride 整体越大，这个实现越慢
- 但一个参数同时改变了多个底层变量，这是**组合效果**
- 要单独验证访存合并，需要固定线程数和循环次数，只改索引公式

### 标签说明

- **measured**: 实测数据（来自 GPU event 或 rocprofv3）
- **reference**: 参考数据（当 profiler 不可用时的 fallback）
- **hypothesis**: 当前假设（需要后续公平对照验证）

## Pass Criteria

本章操作通过标准：

### 必须满足

1. ✅ 能编译并运行 `vector_add_bench`，输出 coalesced 和 linecross 的时间
2. ✅ 确认 `linecross stride=32` 明显慢于 coalesced（约 6.7 倍）
3. ✅ 理解 stride 同时改变了地址排布、循环次数和 Grid Size
4. ✅ 能解读 benchmark 输出：最短时间、有效带宽、正确性
5. ✅ 理解 reference fallback：当 profiler 不可用时，加载已有证据

### 推荐完成（如果 rocprofv3 可用）

6. ⭐ 采集 kernel trace，对比 Grid Size、VGPR、SGPR
7. ⭐ 验证 kernel trace 时间与 benchmark 时间一致
8. ⭐ 观察 stride 扫描曲线，识别整体趋势

### 常见问题排查

| 现象 | 可能原因 | 解决方法 |
|------|----------|----------|
| hipcc 编译失败 | ROCm 未激活 | `source code/part1-profiling/activate-rocm.sh` |
| rocprofv3 不可用 | ROCm 版本 < 7.13 | 使用 reference fallback 数据 |
| 时间差异很大 | 后台负载 | 关闭其他 GPU 任务 |
| CSV 文件为空 | profiler 权限问题 | 检查 `/tmp` 写权限 |
| 正确性检查失败 | 输入规模或 stride 不匹配 | 检查命令行参数 |

## 本章小结

- benchmark 先告诉你“哪个配置更慢”；`rocprofv3 --kernel-trace` 再告诉你“慢在哪个 dispatch”。
- `linecross stride=32` 约为 2.25 ms，明显慢于连续访存版的 0.334 ms。
- 当前 `linecross` 同时改变地址排布、每线程循环次数和 Grid Size，因此不能把 6.7 倍差距全部归因于访存合并。
- 下一步应固定线程数和每线程工作量，只改变索引公式，再重新测量。

---

## 延伸阅读

- [ROCprofiler 文档](https://rocm.docs.amd.com/projects/rocprofiler/en/latest/)
- [HIP Performance Guidelines](https://rocm.docs.amd.com/projects/HIP/en/latest/how-to/performance_guidelines.html)
- [GPUOpen：Memory Coalescing](https://gpuopen.com/learn/gcn-memory-coalescing/)

**下一章**: [第7章 读懂 Roofline 图](./chapter7.ipynb)